# Persistence analysis and real-time vs. revised macroeconomic data

This notebook analyzes the persistence (autocorrelation) of excess bond returns and macroeconomic predictors. It compares overlapping versus non-overlapping return specifications, as well as revised versus real-time macroeconomic data, to formalize the discussion around persistence and validation-induced overfitting.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import utils.base_utils as bu
from utils.forecast_vintages import ForecastVintageMacroStore

sns.set_theme(style='whitegrid', context='talk')

start_date = '1990-01-31'
end_date = '2025-06-30'
MATURITIES = ['24', '36', '48', '60', '84', '120']
maturities_all = [str(i) for i in range(12, 121) if i % 12 == 0]

# Load KR yields
yields = bu.get_yields(type='kr', start=start_date, end=end_date, maturities=maturities_all)
monthly_yields = bu.get_yields(type='kr', start=start_date, end=end_date, maturities=[str(i) for i in range(1, 121)])
forward_rates = bu.get_forward_rates(yields)

# Annual overlapping excess returns
xr = bu.get_excess_returns(yields, horizon=12).dropna()

# Monthly non-overlapping excess returns
monthly_xr = bu.get_excess_returns(monthly_yields, horizon=1).dropna()

# 0. Persistence of forward rates 

In [ ]:
print("AR(1) coefficients for forward rates:")
print(forward_rates[MATURITIES].apply(lambda x: x.autocorr(lag=1))) 

# 1. Persistence of Excess Bond Returns

First, we analyze the autocorrelation function (ACF) of both overlapping (annual) and non-overlapping (monthly) excess returns.

In [ ]:
print("AR(1) coefficients for overlapping annual returns:")
print(xr[MATURITIES].apply(lambda x: x.autocorr(), axis=0))

print("\nAR(1) coefficients for non-overlapping monthly returns:")
print(monthly_xr[MATURITIES].apply(lambda x: x.autocorr(), axis=0))

# 2. Persistence of Macroeconomic Data: Revised vs. Real-Time

We compare the persistence (AR(1) coefficients) of variables between a fully revised FRED-MD dataset and real-time vintages.

In [2]:
fred_md_start_date = pd.to_datetime(start_date) - pd.DateOffset(months=6)
# Load revised FRED-MD data
fred_md_revised = bu.get_fred_data('data/2026-01-MD.csv', start=fred_md_start_date, end=end_date)

# Calculate autocorrelation for revised macro series
revised_acf = fred_md_revised.apply(lambda x: x.autocorr(), axis=0)

print("Top 10 most persistent series in revised data:")
print(revised_acf.nlargest(10))
print("\nSummary statistics of AR(1) across all revised series:")
print(revised_acf.describe())

Top 10 most persistent series in revised data:
PERMIT      0.987467
AAAFFM      0.987365
BAAFFM      0.986492
PERMITS     0.982078
T10YFFM     0.980877
PERMITW     0.972651
HOUST       0.971184
PERMITMW    0.969534
T5YFFM      0.969126
HOUSTS      0.950696
dtype: float64

Summary statistics of AR(1) across all revised series:
count    126.000000
mean       0.145109
std        0.453303
min       -0.609961
25%       -0.179099
50%        0.071891
75%        0.405019
max        0.987467
dtype: float64


# 3. Real-time and revised differences 

In [3]:
eval_dates = pd.date_range(start='1990-01-31', end='2025-06-30', freq='ME')
store = ForecastVintageMacroStore()

group_mapping = {
    # Group 1: Output and Income (16)
    'RPI': 'Output and Income', 'W875RX1': 'Output and Income', 'INDPRO': 'Output and Income', 
    'IPFPNSS': 'Output and Income', 'IPFINAL': 'Output and Income', 'IPCONGD': 'Output and Income', 
    'IPDCONGD': 'Output and Income', 'IPNCONGD': 'Output and Income', 'IPBUSEQ': 'Output and Income', 
    'IPMAT': 'Output and Income', 'IPDMAT': 'Output and Income', 'IPNMAT': 'Output and Income', 
    'IPMANSICS': 'Output and Income', 'IPB51222S': 'Output and Income', 'IPFUELS': 'Output and Income', 
    'CUMFNS': 'Output and Income',
    
    # Group 2: Labor Market (31)
    'HWI': 'Labor Market', 'HWIURATIO': 'Labor Market', 'CLF16OV': 'Labor Market', 
    'CE16OV': 'Labor Market', 'UNRATE': 'Labor Market', 'UEMPMEAN': 'Labor Market', 
    'UEMPLT5': 'Labor Market', 'UEMP5TO14': 'Labor Market', 'UEMP15OV': 'Labor Market', 
    'UEMP15T26': 'Labor Market', 'UEMP27OV': 'Labor Market', 'CLAIMSx': 'Labor Market', 
    'PAYEMS': 'Labor Market', 'USGOOD': 'Labor Market', 'CES1021000001': 'Labor Market', 
    'USCONS': 'Labor Market', 'MANEMP': 'Labor Market', 'DMANEMP': 'Labor Market', 
    'NDMANEMP': 'Labor Market', 'SRVPRD': 'Labor Market', 'USTPU': 'Labor Market', 
    'USWTRADE': 'Labor Market', 'USTRADE': 'Labor Market', 'USFIRE': 'Labor Market', 
    'USGOVT': 'Labor Market', 'CES0600000007': 'Labor Market', 'AWOTMAN': 'Labor Market', 
    'AWHMAN': 'Labor Market', 'CES0600000008': 'Labor Market', 'CES2000000008': 'Labor Market', 
    'CES3000000008': 'Labor Market',
    
    # Group 3: Housing (10)
    'HOUST': 'Housing', 'HOUSTNE': 'Housing', 'HOUSTMW': 'Housing', 'HOUSTS': 'Housing', 
    'HOUSTW': 'Housing', 'PERMIT': 'Housing', 'PERMITNE': 'Housing', 'PERMITMW': 'Housing', 
    'PERMITS': 'Housing', 'PERMITW': 'Housing',
    
    # Group 4: Consumption, Orders, and Inventories (10)
    'DPCERA3M086SBEA': 'Consumption, Orders, and Inventories', 'CMRMTSPLx': 'Consumption, Orders, and Inventories', 
    'RETAILx': 'Consumption, Orders, and Inventories', 'ACOGNO': 'Consumption, Orders, and Inventories', 
    'AMDMNOx': 'Consumption, Orders, and Inventories', 'ANDENOx': 'Consumption, Orders, and Inventories', 
    'AMDMUOx': 'Consumption, Orders, and Inventories', 'BUSINVx': 'Consumption, Orders, and Inventories', 
    'ISRATIOx': 'Consumption, Orders, and Inventories', 'UMCSENTx': 'Consumption, Orders, and Inventories',
    
    # Group 5: Money and Credit (13)
    'M1SL': 'Money and Credit', 'M2SL': 'Money and Credit', 'M2REAL': 'Money and Credit', 
    'BOGMBASE': 'Money and Credit', 'TOTRESNS': 'Money and Credit', 'NONBORRES': 'Money and Credit', 
    'BUSLOANS': 'Money and Credit', 'REALLN': 'Money and Credit', 'NONREVSL': 'Money and Credit', 
    'CONSPI': 'Money and Credit', 'DTCOLNVHFNM': 'Money and Credit', 'DTCTHFNM': 'Money and Credit', 
    'INVEST': 'Money and Credit',
    
    # Group 6: Interest and Exchange Rates (22)
    'FEDFUNDS': 'Interest and Exchange Rates', 'CP3Mx': 'Interest and Exchange Rates', 
    'TB3MS': 'Interest and Exchange Rates', 'TB6MS': 'Interest and Exchange Rates', 
    'GS1': 'Interest and Exchange Rates', 'GS5': 'Interest and Exchange Rates', 
    'GS10': 'Interest and Exchange Rates', 'AAA': 'Interest and Exchange Rates', 
    'BAA': 'Interest and Exchange Rates', 'COMPAPFFx': 'Interest and Exchange Rates', 
    'TB3SMFFM': 'Interest and Exchange Rates', 'TB6SMFFM': 'Interest and Exchange Rates', 
    'T1YFFM': 'Interest and Exchange Rates', 'T5YFFM': 'Interest and Exchange Rates', 
    'T10YFFM': 'Interest and Exchange Rates', 'AAAFFM': 'Interest and Exchange Rates', 
    'BAAFFM': 'Interest and Exchange Rates', 'TWEXAFEGSMTHx': 'Interest and Exchange Rates', 
    'EXSZUSx': 'Interest and Exchange Rates', 'EXJPUSx': 'Interest and Exchange Rates', 
    'EXUSUKx': 'Interest and Exchange Rates', 'EXCAUSx': 'Interest and Exchange Rates',
    
    # Group 7: Prices (20)
    'WPSFD49207': 'Prices', 'WPSFD49502': 'Prices', 'WPSID61': 'Prices', 'WPSID62': 'Prices', 
    'OILPRICEx': 'Prices', 'PPICMM': 'Prices', 'CPIAUCSL': 'Prices', 'CPIAPPSL': 'Prices', 
    'CPITRNSL': 'Prices', 'CPIMEDSL': 'Prices', 'CUSR0000SAC': 'Prices', 'CUSR0000SAD': 'Prices', 
    'CUSR0000SAS': 'Prices', 'CPIULFSL': 'Prices', 'CUSR0000SA0L2': 'Prices', 'CUSR0000SA0L5': 'Prices', 
    'PCEPI': 'Prices', 'DDURRG3M086SBEA': 'Prices', 'DNDGRG3M086SBEA': 'Prices', 'DSERRG3M086SBEA': 'Prices',
    
    # Group 8: Stock Market (4)
    'S&P 500': 'Stock Market', 'S&P div yield': 'Stock Market', 'S&P PE ratio': 'Stock Market', 'VIXCLSx': 'Stock Market'
}

In [60]:
rows = []

for d in eval_dates:
    panel = store.panel_for_forecast_date(d, start=start_date)
    rt = panel.transformed

    # Final revised data must be aligned to same date window
    revised_same_window = fred_md_revised.loc[rt.index, rt.columns]

    for col in rt.columns:
        # Find the overlapping non-NA indices
        common_idx = rt[col].dropna().index.intersection(
            revised_same_window[col].dropna().index
        )

        missing_rt = rt[col].isna().mean()
        missing_rev = revised_same_window[col].isna().mean()

        # Safety check: We need at least 3 points to calculate standard dev and lag-1 autocorrelation
        if len(common_idx) < 3:
            rows.append({
                "eval_date": d,
                "series": col,
                "rt_mean": np.nan,
                "rev_mean": np.nan,
                "revision_mean": np.nan,
                "rt_sd": np.nan,
                "rev_sd": np.nan,
                "revision_sd": np.nan,
                "rt_acf1": np.nan,
                "rev_acf1": np.nan,
                "acf1_diff": np.nan,
                "n_obs": len(common_idx),
                "missing_rt": missing_rt,
                "missing_rev": missing_rev,
            })
            continue

        rt_common = rt.loc[common_idx, col]
        rev_common = revised_same_window.loc[common_idx, col]
        revision = rev_common - rt_common

        rt_sd = rt_common.std()
        rev_sd = rev_common.std()
        
        # Safety check: Prevent division by zero in autocorrelation if a series is flat
        rt_acf1 = rt_common.autocorr(lag=1) if rt_sd > 0 else np.nan
        rev_acf1 = rev_common.autocorr(lag=1) if rev_sd > 0 else np.nan
        
        # 1. Correlation between RT and Rev
        correlation = rt_common.corr(rev_common, method='kendall') if (rt_sd > 0 and rev_sd > 0) else np.nan
        # 2. Mean Absolute Error of the revision
        mae_revision = revision.abs().mean()

        rows.append({
            "eval_date": d,
            "series": col,
            "rt_mean": rt_common.mean(),
            "rev_mean": rev_common.mean(),
            "revision_mean": revision.mean(),
            "rt_sd": rt_sd,
            "rev_sd": rev_sd,
            "revision_sd": revision.std(),
            "mae_revision": mae_revision,
            "correlation": correlation,          
            "rt_acf1": rt_acf1,
            "rev_acf1": rev_acf1,
            "acf1_diff": rev_acf1 - rt_acf1 if (not np.isnan(rt_acf1) and not np.isnan(rev_acf1)) else np.nan,
            "n_obs": len(common_idx),
            "missing_rt": missing_rt,
            "missing_rev": missing_rev,
        })

vintage_stats = pd.DataFrame(rows)

/home/ulrikts/Documents/NTNU/TIO4900-Replication/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/ulrikts/Documents/NTNU/TIO4900-Replication/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [57]:
summary_by_series = (
    vintage_stats
    .groupby("series")
    .agg(
        mean_rt_acf1=("rt_acf1", "mean"),
        mean_rev_acf1=("rev_acf1", "mean"),
        mean_acf1_diff=("acf1_diff", "mean"),
        mean_rt_sd=("rt_sd", "mean"),
        mean_rev_sd=("rev_sd", "mean"),
        mean_revision_sd=("revision_sd", "mean"),
        mean_mae=("mae_revision", "mean"),         # <-- ADDED
        mean_corr=("correlation", "mean")          # <-- ADDED
    )
)

summary_by_series['Group'] = summary_by_series.index.map(group_mapping)

# Calculate ratios
summary_by_series['revision_noise_ratio'] = summary_by_series['mean_revision_sd'] / summary_by_series['mean_rt_sd']
summary_by_series['volatility_ratio'] = summary_by_series['mean_rev_sd'] / summary_by_series['mean_rt_sd']

# Create the final clean table
group_summary = summary_by_series.groupby('Group').agg(
    Avg_RT_AR1=('mean_rt_acf1', 'mean'),
    Avg_AR1_Diff=('mean_acf1_diff', 'mean'),
    Avg_Volatility_Ratio=('volatility_ratio', 'mean'),
    Avg_Revision_Noise=('revision_noise_ratio', 'mean'),
    Avg_Correlation=('mean_corr', 'mean')
).round(3)

print(group_summary)

                                      Avg_RT_AR1  Avg_AR1_Diff  \
Group                                                            
Consumption, Orders, and Inventories      -0.067         0.002   
Housing                                    0.865        -0.001   
Interest and Exchange Rates                0.573        -0.005   
Labor Market                               0.125         0.022   
Money and Credit                          -0.284         0.005   
Output and Income                          0.007         0.022   
Prices                                    -0.330         0.026   
Stock Market                               0.426         0.001   

                                      Avg_Volatility_Ratio  \
Group                                                        
Consumption, Orders, and Inventories                 1.002   
Housing                                              0.987   
Interest and Exchange Rates                          0.999   
Labor Market                 

In [ ]:
# plot all Money and Credit series
money_credit_series = [s for s, g in group_mapping.items() if g == 'Money and Credit']
plt.figure(figsize=(15, 10))
for series in money_credit_series:  
    plt.plot(vintage_stats[vintage_stats['series'] == series]['eval_date'], vintage_stats[vintage_stats['series'] == series]['correlation'], label=series)